- Useful reading: 
    - Statistics, Data mining and Machine Learning in Astronomy (Ivezic, Connolly, VanderPlas & Gray 2014)
- Want to learn more about Bayesian stats? **MAP6469**
    

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import astropy.io.fits as fits
import gaussian as g
import radprof as R
import disk as d

### Classical vs Bayesian statistics:

- Classical statistics:
    - find probability/frequency $p(x)$ with which x occurs if we do similar experiments
    - probabilities are relative frequencies of events; objective properties of the real world
    - parameters are fixed, unknown constants (e.g., 4/7 coin tosses are head)
    - statistical procedures have well-defined frequency properties (e.g., 95% confidence interval brackets the true value of the parameter with a limiting frequency of at least 95%)
    
- Bayesian statistics:
    - find probability of x given a 'truth'/observed happening y $p(x|y)$ 
    - probabilities describe the degree of subjective belief, not the limiting frequency
    - we can make probability statements about not only the data, but also model parameters and models themselves
    - inferences about a parameter are made by producing its probability distribution
        - quantifies the uncertainty of our knowledge about that parameter
        - point estimates, such as expectation value, may then be readily extracted from this distribution



### Bayesian statistics

- you have observed data D (e.g., flux from target star over a full rotational period) and want to infere from this model M (e.g., find surface coverage of magnetic star spots)

    - use conditional probabilities (form: $ p(A|B)$):
        - co-dependent quantities A and B: $p(A|B)p(B) = p(B|A)p(A)$
        - probability distribution of A: $p(A) = \int p(A|B) p(B) dB $ 
        - combining these $\rightarrow$ Bayes' theorem $$ p(B|A) =\frac{p(A|B) p(B)}{p(A)} = \frac{p(A|B) p(B)}{\int p(A|B) p(B) dB} $$
    - Steps to take:
        - define uncertainty of the wanted/unknown model M through a prior distribution $p(M)$
        - assume a reasonable distribution for the sampling distribution of your data given y $p(D|M)$
        - calculate posterior distribution $p(M|D)$ following Bayes' theorem: $p(M|D) = \frac{p(D|M) p(M)}{p(D)}$
            - often, you will see Bayes' theorem as $p(M,\vec{p}|D,I) = \frac{p(D|M,\vec{p},I) p(M\vec{p}|I)}{p(D|I)}$ with $\vec{p}$ the parameters of the model M you want to fit, $I$ is the prior, $D$ your data
    
            - you have an initial belief about your model that you combine with new data to get an improved belief about the model
            - $p(M,\vec{p}|D,I)$ is the posterior distribution (posterior pdf) of model $M$ and parameters $\vec{p}$ given data $D$ and prior information $I$
        
    
    
- application: MCMC

### Markov Chain Monte Carlo (MCMC):


- Monte Carlo:
    - repeated random sampling from your code to get a numerical result
        - numerical integrations of functions that can be graphed and don’t have simple analytic solution 
        - simulations of random variables using random samples from a uniform distribution 
        - estimation of uncertainties in the best-fit parameters of analytical models used to fit data 

- Markov chains:
    - chains of variables where every step depends only on the close-by previous steps ('forgets' about steps further away/initial steps; $p(\theta_{i+1}|\{\theta_i\}) = p(\theta_{i+1}|\theta_i) $)
    
    
- MCMC: random walk/ change of a variable that you want to fit using Bayesian stats to define whether step will happen or not
    - allow a parameter to vary in a 'random' way (Monte Carlo) but every step depends *only* on the step before (Markov Chain) and the ones that came before that
    - there is a probability that a given step can happen; figure out which steps are visited not too often, which are happening all the time
    - try multiple starting points; make sure you have enough steps in your chain to lose info of the start *and* be able to explore enough parameter space for each variable
    - Bayesian priors & posterior distributions <br> <img src="mcmc_chains_iter.png" width=550><img src="emcee_corner_plot.png" width=350>([example_MCMC_chains](https://atlas.cancer.org.au/developing-a-cancer-atlas/Chapter_3.html) and [emcee](https://emcee.readthedocs.io/en/stable/tutorials/line/) )

In [ ]:
#if you don't have it already you can get it with
#conda update conda
#conda install -c conda-forge emcee

#(or pip see: https://emcee.readthedocs.io/en/stable/user/install/ )

#and if you run it you will want corner.py for corner plots too:
#pip install corner

#and to show the progress bar you need:
#pip install tqdm


In [ ]:
import emcee
import corner

In [ ]:
#example based on emcee's https://emcee.readthedocs.io/en/stable/tutorials/quickstart/#quickstart
#define the log probability funct for a gaussian:
def log_prob(x, mu, cov):
    diff = x - mu
    return -0.5 * np.dot(diff, np.linalg.solve(cov, diff))

In [ ]:
#define your parameters
ndim = 5

np.random.seed(42)
means = np.random.rand(ndim)

cov = 0.5 - np.random.rand(ndim**2).reshape((ndim, ndim))
cov = np.triu(cov)
cov += cov.T - np.diag(cov.diagonal())
cov = np.dot(cov, cov)

In [ ]:
#define the number of your walkers and a starting point:
nwalkers = 32
p0 = np.random.rand(nwalkers, ndim)

In [ ]:
#call the sampler
sampler = emcee.EnsembleSampler(nwalkers, ndim, log_prob, args=[means, cov])

In [ ]:
#run it and discard a few steps at the start to 'lose memory' of where you started from

# Run the MCMC
sampler.run_mcmc(p0, 420000, progress=True)

# Get the samples
samples = sampler.get_chain(discard=1000, thin=15, flat=True)

In [ ]:
plt.hist(samples[:, 0], 100, color="k", histtype="step")
plt.xlabel(r"$\theta_1$")
plt.ylabel(r"$p(\theta_1)$")
plt.gca().set_yticks([]);

In [ ]:
#perfect gaussian :)



In [ ]:
#another example from https://emcee.readthedocs.io/en/stable/tutorials/line/:


#make your dataset:
#=================================
np.random.seed(123)

# Choose the "true" parameters.
m_true = -0.9594
b_true = 4.294
f_true = 0.534

# Generate some synthetic data from the model
#=================================
N = 50
x = np.sort(10 * np.random.rand(N))
yerr = 0.1 + 0.5 * np.random.rand(N)
y = m_true * x + b_true
y += np.abs(f_true * y) * np.random.randn(N)
y += yerr * np.random.randn(N)

#do a least square fit:
#=================================
plt.errorbar(x, y, yerr=yerr, fmt=".k", capsize=0)
x0 = np.linspace(0, 10, 500)
plt.plot(x0, m_true * x0 + b_true, "k", alpha=0.3, lw=3)
plt.xlim(0, 10)
plt.xlabel("x")
dm = plt.ylabel("y")

A = np.vander(x, 2)
C = np.diag(yerr * yerr)
ATA = np.dot(A.T, A / (yerr**2)[:, None])
cov = np.linalg.inv(ATA)
w = np.linalg.solve(ATA, np.dot(A.T, y / yerr**2))
print("Least-squares estimates:")
print("m = {0:.3f} ± {1:.3f}".format(w[0], np.sqrt(cov[0, 0])))
print("b = {0:.3f} ± {1:.3f}".format(w[1], np.sqrt(cov[1, 1])))

plt.errorbar(x, y, yerr=yerr, fmt=".k", capsize=0)
plt.plot(x0, m_true * x0 + b_true, "k", alpha=0.3, lw=3, label="truth")
plt.plot(x0, np.dot(np.vander(x0, 2), w), "--k", label="LS")
plt.legend(fontsize=14)
plt.xlim(0, 10)
plt.xlabel("x")
plt.ylabel("y");

#define your log likelihood func:
#=================================

def log_likelihood(theta, x, y, yerr):
    m, b, log_f = theta
    model = m * x + b
    sigma2 = yerr**2 + model**2 * np.exp(2 * log_f)
    return -0.5 * np.sum((y - model) ** 2 / sigma2 + np.log(sigma2))

#find the numerical optimum of this likelihood function
#=================================
from scipy.optimize import minimize

np.random.seed(42)
nll = lambda *args: -log_likelihood(*args)
initial = np.array([m_true, b_true, np.log(f_true)]) + 0.1 * np.random.randn(3)
soln = minimize(nll, initial, args=(x, y, yerr))
m_ml, b_ml, log_f_ml = soln.x

print("Maximum likelihood estimates:")
print("m = {0:.3f}".format(m_ml))
print("b = {0:.3f}".format(b_ml))
print("f = {0:.3f}".format(np.exp(log_f_ml)))

plt.errorbar(x, y, yerr=yerr, fmt=".k", capsize=0)
plt.plot(x0, m_true * x0 + b_true, "k", alpha=0.3, lw=3, label="truth")
plt.plot(x0, np.dot(np.vander(x0, 2), w), "--k", label="LS")
plt.plot(x0, np.dot(np.vander(x0, 2), [m_ml, b_ml]), ":k", label="ML")
plt.legend(fontsize=14)
plt.xlim(0, 10)
plt.xlabel("x")
plt.ylabel("y");

#define your prior
#=================================
def log_prior(theta):
    m, b, log_f = theta
    if -5.0 < m < 0.5 and 0.0 < b < 10.0 and -10.0 < log_f < 1.0:
        return 0.0
    return -np.inf

#use them to make your probability func:
#=================================
def log_probability(theta, x, y, yerr):
    lp = log_prior(theta)
    if not np.isfinite(lp):
        return -np.inf
    return lp + log_likelihood(theta, x, y, yerr)


#run it!
#=================================
pos = soln.x + 1e-4 * np.random.randn(32, 3)
nwalkers, ndim = pos.shape

sampler = emcee.EnsembleSampler(
    nwalkers, ndim, log_probability, args=(x, y, yerr)
)
sampler.run_mcmc(pos, 5000, progress=True);

In [ ]:
#plot chains
#=================================
fig, axes = plt.subplots(3, figsize=(10, 7), sharex=True)
samples = sampler.get_chain()
labels = ["m", "b", "log(f)"]
for i in range(ndim):
    ax = axes[i]
    ax.plot(samples[:, :, i], "k", alpha=0.3)
    ax.set_xlim(0, len(samples))
    ax.set_ylabel(labels[i])
    ax.yaxis.set_label_coords(-0.1, 0.5)

axes[-1].set_xlabel("step number")


#get an estimate of how soon the chains burn-in (the integrated autocorrelation time)
#=================================

tau = sampler.get_autocorr_time()
print('Tau:',tau)



In [ ]:
#discard a bit more than double that number to forget where you start from:
#=================================

flat_samples = sampler.get_chain(discard=100, thin=15, flat=True)


#and plot your corner plot:
#=================================

fig = corner.corner(
    flat_samples, labels=labels, truths=[m_true, b_true, np.log(f_true)]
)

In [ ]:
from IPython.display import display, Math

for i in range(ndim):
    mcmc = np.percentile(flat_samples[:, i], [16, 50, 84])
    q = np.diff(mcmc)
    txt = "\mathrm{{{3}}} = {0:.3f}_{{-{1:.3f}}}^{{{2:.3f}}}"
    txt = txt.format(mcmc[1], q[0], q[1], labels[i])
    display(Math(txt))

In [ ]:
#compare with least sq estimates:
#m = -1.104 ± 0.016
#b = 5.441 ± 0.091

In [ ]:
#m_true = -0.9594
#b_true = 4.294
#f_true = 0.534

### Let's also see the application of [ultranest](https://johannesbuchner.github.io/UltraNest/index.html) in fitting a model light curve

In [ ]:
import ultranest
from ultranest.plot import cornerplot
import ultranest.stepsampler

In [ ]:
#let's make a dataset:
from numpy import sin, pi

def sine_model2(t, B, A1, P1, t1, A2, P2, t2):
    return A1 * sin((t / P1 + t1) * 2 * pi) + A2 * sin((t / P2 + t2) * 2 * pi) + B


np.random.seed(42)

n_data = 50

# time of observations
t = np.random.uniform(0, 5, size=n_data)
# measurement values
yerr = 1.0*0.001
y = np.random.normal(sine_model2(t, B=1.0, A1=0.002, P1=5., t1=0, A2=0.02, P2=1.2, t2=1.2), yerr)




In [ ]:
#plot our "observations"
plt.figure( figsize = (10,5))
dm = plt.errorbar( t, y, yerr = yerr, linestyle='',marker = 'o', color='xkcd:bright blue')

In [ ]:
#set our model up
parameters2 = ['B', 'A1', 'P1', 't1', 'A2', 'P2', 't2']

def prior_transform2(cube):
    # the argument, cube, consists of values from 0 to 1
    # we have to convert them to physical scales

    params = cube.copy()
    # let background level go from -10 to +10
    params[0] = cube[0] * 20 - 10
    # let amplitude go from 0.1 to 100
    params[1] = 10**(cube[1] * 3 - 1)*1e-3
    # let period go from 0.3 to 30
    params[2] = 10**(cube[2] * 1.4) #2
    # let time go from 0 to 1
    params[3] = cube[3]

    # let amplitude go from 0.01 to 100
    params[4] = 10**(cube[4] * 3 - 1)*1e-3
    # let period go from 0.3 to 30
    params[5] = 10**(cube[5] * 1.4) #2
    # let time go from 0 to 1
    params[6] = cube[6]
    return params

def log_likelihood2(params):
    # unpack the current parameters:
    B, A1, P1, t1, A2, P2, t2 = params

    # avoid unnecessary multiple solutions:
    #    force ordering by period from large to small
    if P1 < P2:
        # instead of returning a very low number:
        ## return -1e300
        # which would give a likelihood plateau causing some loss of live points
        # we give a slope towards the "good" parameter space:
        return -1e300 * abs(P1 - P2)

    # compute for each x point, where it should lie in y
    y_model = sine_model2(t, B=B, A1=A1, P1=P1, t1=t1, A2=A2, P2=P2, t2=t2)
    # compute likelihood
    loglike = -0.5 * (((y_model - y) / yerr)**2).sum()

    return loglike

In [ ]:
sampler2 = ultranest.ReactiveNestedSampler(
    parameters2,
    log_likelihood2,
    prior_transform2,
    wrapped_params=[False, False, False, True, False, False, True],
)



In [ ]:
nsteps = 4 * len(parameters2)
# create step sampler:
sampler2.stepsampler = ultranest.stepsampler.SliceSampler(
    nsteps=nsteps,
    generate_direction=ultranest.stepsampler.generate_mixture_random_direction,
    # adaptive_nsteps=False,
    # max_nsteps=400
)

# run again:
result2 = sampler2.run(min_num_live_points=2400)
sampler2.print_results()

In [ ]:
plt.figure()
cornerplot(result2,title_fmt='.4f')

## Some coding tips for the end of the semester :) 

### Sometimes your code will be taking a long time to run. What do you do?
- Make sure that you avoid a lot of I/O  

- if possible, consider where code runs; running the code if possible/appropriate at GPUs will help speed it up

- Parallelize code

- Avoid repeating work on subproblems that has already been performed (--> note that if too complex problems, recalling result from memory can end up adding more time...)

- Sometimes, the order of performing specific calculations can also affect the speed of your code 

- Chop up the parameter space and show that some parts can be ignored or approximated during the computation 

- For fitting data (think ML) chose a subset of the points that yield higher leverage or somehow more effectively represent the information in the whole data set

- transform/ decompose complicated functions into simpler ones (simplify your problem -- reduce dimensionality - see below)



###  We will see a very basic multiprocessing example (see also the relative python library [here](https://docs.python.org/3/library/multiprocessing.html) )


In [ ]:
def my_sin( n ) :
    """makes a sin function from array x; where x = np.arange( 1, 10000, n)
    Input : n
    Output: sin( 2* pi * x +4 ) """
    
    x = np.linspace( 1, 10000, n ) 
    y = np.sin( 2 * np.pi * x )
    
    return y

In [ ]:
#import the library
import multiprocessing


#figure out how many cpus you have:
num_procs = multiprocessing.cpu_count()
print( num_procs )

In [ ]:
### Practice it on a small sample:

In [ ]:
%%time
aa = my_sin( 100000)
qq = np.mean( aa ) 

#CPU times: user 4.24 ms, sys: 18.3 ms, total: 22.5 ms
#Wall time: 24.3 ms

In [ ]:
def sin_multiprocessing( n ):
    """Split the sin into num_procs pieces."""
    
    m   = multiprocessing.cpu_count()
    pool = multiprocessing.Pool( m )
    results = pool.map(my_sin,  [int(n/m)]*m) 
    pool.close()
    return np.mean( results )

In [ ]:
%%time
sin_multiprocessing( 100000 )
#CPU times: user 25.8 ms, sys: 110 ms, total: 135 ms
#Wall time: 206 ms

In [ ]:
# Now we have a larger sample 

In [ ]:
%%time
aa = my_sin( 300000000)
qq = np.mean( aa ) 

# CPU times: user 4.58 s, sys: 7.64 s, total: 12.2 s
# Wall time: 14.1 s

In [ ]:
%%time
sin_multiprocessing(300000000)
# CPU times: user 2.87 s, sys: 5.23 s, total: 8.1 s
# Wall time: 12.7 s

### You have the following data:  m1 =  [1, 9, 2, 5, 7, 5]  and  m2 =  [12, 13, 3, 5, 88, -4]  where every element of m1 has a corresponding value in m2 (so 1 and 12 are related, 9 and 13 etc). Sort lists m1 and m2 based on the sorting of m1 from min to max.

In [ ]:
m1 = [1, 9, 2, 5, 7, 5]
m2 = [12, 13, 3, 5, 88, -4] 


In [ ]:
#you might need to install this first (pip install more-itertools)
from more_itertools import sort_together

In [ ]:
s = sort_together([m1, m2])[1]
print(s)

### or, as we have seen before in this class, if they are numpy arrays:

In [ ]:
m1b = np.array( [1, 9, 2, 5, 7, 5] )
m2b = np.array( [12, 13, 3, 5, 88, -4] )

In [ ]:
inds = np.argsort( m1b )
s2   = m2b[ inds ]
print( s2 )

### If you want to look into different plotting for your statistical results there is also [seaborn](https://seaborn.pydata.org/)

### ICA. Read in file PSF_NIRCam_in_flight_opd_filter_F212N.fits that contains the PSF for the narrowband filter at a wavelength of 2.12 µm for the [JWST/NIRCam instrument](https://jwst-docs.stsci.edu/jwst-near-infrared-camera/nircam-instrumentation/nircam-filters). 

- A. read in the file's second "extension". Note that some FITS files have multiple images embedded in them, and the usual format is to put those additional images in "extensions" to the file, with their own headers (so you would have hdu[0].header, hdu[1].header etc). The second extension is a 1288-by-1288 image of the PSF (check that that is correct!). Display the image with a stretch such that you can see some of its structure. Make a publication ready plot and save it.

- B.
    - 1. make a plot of the radial profile from the center of the target. Note that the centroid of the image is at x=643.5, y=643.5 (although you can confirm this with photutil's centroid_com function if you want). Also note that this image is an idealized PSF and so the value of the sky background can be considered to be zero, i.e. the net counts equals the gross counts.
    - 2. make a plot of the enclosed flux Fencl -- i.e., the net counts -- as a function of aperture radius ρ like we did in the previous ICA (SNR class; use disk.py), except this time have the ρ run from 1 to 100 pixels in steps of 1 pixel. Save that plot. 
    
- C. Display this plot in a slightly different way. Let's call ρ = 40 pixels our fiducial aperture radius, ρ0, based on the fact that, say, we use that aperture radius when measuring the instrumental magnitudes of reference stars. The aperture correction $m_\mathrm{ac}$ as a function of aperture radius that's used on the source of interest is just: $$ m_\mathrm{ac}(\rho) = -2.5 \log\frac{F_\mathrm{encl}(\rho)}{F_\mathrm{encl}(\rho_0)}$$ Make a plot of $m_\mathrm{ac}$ as a function of $\rho$, and save that plot.


- D. Repeat the procedure for Part B but for a finer grid of ρ that goes from 1 to 10 pixels in steps of 0.1 pixels. Compare with the old plot. Save that plot. Moral: You should notice that the plot doesn't increase smoothly, but has steps, especially at the smaller aperture radii. This is a manifestation of the pixelization problem, where a PSF is not a perfectly-sampled function but is sampled into a bunch of square pixels. The steps in the plot come from this situation: suppose you ask for a slightly larger aperture, but there are no new additional pixels for which the center of the pixel is now within the threshold. Then the total enclosed flux won't change because there are no new pixels to include!


- one way around the pixelization problem is to calculate what fraction of the area of a pixel that's on the aperture border is actually within the aperture, and then include the same fraction of the counts in that pixel. The schematic below demonstrates this idea. <br> <img src="subpixelization_yanb.png" width=450><br> <br>  On the left is a 3-by-3 subarray of an image, with each pixel labelled with a letter, A through I. The black dot in each pixel marks the center of that pixel. A part of the circular aperture cuts through this subarray; that is the curve in the diagram. Pixels B, C, and F are entirely within the aperture, and pixels A, D, E, H, and I are on the boundary. Pixel G is entirely outside the aperture. With our standard proceduce, we windup including pixels A, B, C, E, F, and I since the centers of those pixels are within the aperture. So the enclosed gross counts will include all the counts from each of those pixels, despite the fact that really only part of the flux within pixels A, E, and I ought to be included. Furthermore, our method will totally leave out any flux in the D and H pixels, even though at least some of the flux ought to be included too.<br><br> So to get around this problem, one method is to subpixelize each pixel. On the right is a blow-up of pixel D, and the extra grid on the pixel represents a 10-by-10 subpixelization of the pixel. In other words we've broken up the pixel into 100 smaller subpixels. With this gridding, it's clear via inspection by eye that approximately 20 subpixels of area are within the aperture. (Just count up the squares, and make an estimate with the partial ones.) So in pixel D, we should be including 20% of the pixel's counts when we calculate our gross counts. One could do a similar eyeball assessment with pixels A, E, H, and I (the other pixels on the boundary), although the real goal would be to generalize this procedure in code. (It should be pointed out that this is pretty much the most simplistic higher-order correction to aperture photometry there is. There are more sophisticated techniques out there, although usually for the accuracy we're going for (~1%) this correction is sufficient.)

- OK so how to do this in code? There are two tasks: Make a new, expanded radius array, and make a new, expanded image array. All we need to keep track of is the coordinates of the centroid in this larger array. Let the coordinates of the centroid in the original array be $x_c$ and $y_c$, let the size of the original array be $n_y$-by-$n_x$, and let the subpixelization factor be $f$. Then the new expanded array has a size of $f n_y$-by-$f n_x$, and the coordinates of the centroid in this new, expanded array -- call those $x_\mathrm{c,exp}$ and $y_\mathrm{c,exp}$ -- are simply: $$x_\mathrm{c,exp} = x_\mathrm{c}f+\frac{1}{2}(f-1) $$ and $$y_\mathrm{c,exp} = y_\mathrm{c}f+\frac{1}{2}(f-1) $$ <br><br>That second term for each coordinate comes from the fact that pixel location (0,0) is not actually at the lower-left corner of the image. Location (0,0) is the middle of the lower-left pixel, but the lower-left corner itself is at coordinates (-0.5,-0.5). So if you take the (0,0) pixel and expand it into, say, 10-by-10 subpixels, then the (0,0) location maps in the new array to location (4.5,4.5), not to (5,5). (You can convince yourself of this by looking at the graphic of pixel D above.) Or to put it another way, the new lower-left subpixel in the new array doesn't go from (0,0) to (0.1,0.1) in the old array coordinates, it goes from (-0.5,-0.5) to (-0.4,-0.4). This framework means you can use the same general approach as before while remembering how the new $\rho$ array connects to the original pixels. 


- E. Determine the centroid position using $f$=10. 
    - Then convert the image array into subpixels. For this, we can use scipy's [ndimage.zoom](https://docs.scipy.org/doc/scipy/reference/generated/scipy.ndimage.zoom.html) function. In this case, we simply want to expand the array without any interpolation. As before, use f=10. There are a lot of options with the zoom function, so in this case: We don't want interpolation to the zoomed image, so set "order=0". We want the function to interpret the full width of the array so set "grid_mode=True". We want the outer edge of the array to be the same value as the next pixel in, so set "mode='nearest'". Furthermore, the output from zoom should be normalized in order to preserve the total counts, so divide your output array by 100. That way each subpixel has 1/100th of the original flux (or in other words you want the total counts from each set of 10-by-10 subpixels to add up to the counts in the original pixel).


- F. Do the same procedure as you did before to make the plot in Part A, except make use of the expanded PSF image and the finner grid from part D. Plot it against the other $F_\mathrm{encl}(\rho)$ line. You should find that the curve is now smooth. Save that plot.